## <a href="https://cursos.alura.com.br/course/langchain-desenvolva-agentes-inteligencia-artificial/task/161398"><b>Langchain Agentes - Implementando um agente com sinergia entre raciocínio e ação (ReAct)</b></a><br/>

<b>Objetivos:<br/>
<ol>
    <li>Que outro prompt eu posso utilizar, se eu não tiver o openai functions na criação do agente, ou seja, esteja utlizando outra LLM ?</li>
</ol>

<b>Abordagem e metodologia</b><br/><br/>
O framework ReAct promove uma fusão entre raciocínio detalhado e ações práticas dentro de um fluxo de trabalho interativo, permitindo que modelos de linguagem atuem de forma adaptativa e contextual. Esse método é particularmente valioso em domínios que exigem verificação de fatos e respostas informadas por dados atualizados.

In [1]:
#%pip install -r requirements.txt

### <b>FERRAMENTA DadosDeEstudante</b>

<ol>
    <b><li>Constatar que não é necessário invocar a LLM para se obter o nome do estudante na ferramenta DadosDeEstudante, porque a descrição informa isso.</li></b>
    <ul>
        <b><li>estudante = input</li></b><br/>
    </ul>
    <b><li>Melhorar o Prompt para se evitar alucinações, dados inventados ou erros de retorno.</li></b><br/>
    <ul>    <li>description : str = """ 
                            - Essa ferramenta extrai o histórico e preferências de um estudante, de acordo com o seu histórico.<br/>
                            <b>- O parâmetro de entrada desta ferramenta, deve ser o nome do estudante.</b>
                        """ # Descrição da ferramenta</li>
    </ul> 
</ol>

In [2]:
from pydantic import BaseModel, Field
from langchain.tools import BaseTool
from pandas import read_csv
import json

# FERRAMENTA DADOS DE ESTUDANTE
class DadosDeEstudante(BaseTool): # ESTENDE BaseTool
    
    # TODA FERRAMENTA PRECISA TER ESSES ATRIBUTOS
    name: str = "dados_de_estudante" # Nome da ferramenta
    description : str = """ 
                            - Essa ferramenta extrai o histórico e preferências de um estudante, de acordo com o seu histórico.
                            - Passe para essa ferramenta como argumento o nome do estudante. 
                        """ # Descrição da ferramenta
                            # Melhorar o Prompt para se evitar alucinações, dados inventados ou erros de retorno.
    
    def __init__(self): 
        super().__init__() # PARA NÃO SOBRESCREVER O CONSTRUTOR DA CLASSE MÃE BaseTool
                
        print('Inicializando ferramenta Dados de Estudante')      
    
    def __busca_dados_de_estudante(self,estudante:str) -> str:
        
        dfestudantes = read_csv("documentos/estudantes.csv")
        
        dados_estudante = dfestudantes.loc[dfestudantes['USUARIO'] == estudante]
        
        if dados_estudante.empty:
            return f"Desculpe, não encontrei dados para o estudante '{estudante}'. Por favor, verifique o nome e tente novamente."
        
        json_estudante = json.dumps(dados_estudante.to_dict(orient='records')[0]) # MELHOR FORMATO PARA RETORNAR OS DADOS ENTRE FERRAMENTAS
        
        return json_estudante
        
                
    # CONTRATO DA FERRAMENTA - O QUE ELA FAZ
    def _run(self, input: str) -> str:     
        
        # MODIFICAÇÃO
        estudante = input
                
        print(f'\nRetorno estudante {estudante} da LLM')        
        
        dados = self.__busca_dados_de_estudante(estudante.lower())
        print('JSON retornado pelo método busca_dados_de_estudante:', dados)
               
        return dados


In [3]:
from typing import List

class Nota(BaseModel):
    area_de_conhecimento: str = Field(description="Área de conhecimento da nota")
    nota: float = Field(description="Nota obtida pelo estudante nessa área de conhecimento")
    
class ExtratorPerfilAcademicoDeEstudante(BaseModel):
    
    nome:str = Field(description="Nome do estudante")
    ano_de_conclusao: int = Field(description="Ano de formatura")
    notas: List[Nota] = Field(description="Lista de notas para cada área de conhecimento") # O Nota dentro da lista, é a classe criada acima, que 
                                                                                           # formata o dicionário de notas
                                                                                           
    resumo: str = Field(description="Resumo das principais características desse estudante de forma a torná-lo único e um ótimo potencial estudante para faculdades. Só esse estudante tem bla bla bla")

### <b>CRIAÇÃO DE FERRAMENTAS</b>

<b>1) Criação da Ferramenta Perfil Acadêmico</b>

Única ferramenta que está invocando LLM.

In [4]:
from langchain.tools import BaseTool
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
    
class PerfilAcademico(BaseTool): # ESTENDE BaseTool
    
    llm:ChatOpenAI = None
    
    # TODA FERRAMENTA PRECISA TER ESSES ATRIBUTOS
    name: str = "perfil_academico" # Nome da ferramenta
    description : str = """                                                       
                            - Esta ferramenta utiliza os dados do estudante para gerar um perfil acadêmico detalhado.  
                            - Não consigo obter os dados do estudante sozinho. **NUNCA** utilizar somente o nome, execute a ferramenta **dados_de_estudante** 
                            antes para obter os dados.                                         
                        """ # Descrição da ferramenta  
    
    def __init__(self,llm:ChatOpenAI):
        super().__init__() # PARA NÃO SOBRESCREVER O CONSTRUTOR DA CLASSE MÃE BaseTool
        self.llm = llm   
        print('Inicializando ferramenta Perfil Acadêmico')
        
                
    # CONTRATO DA FERRAMENTA - O QUE ELA FAZ
    #
    # JÁ QUE SE TRATA DA ENTRADA DOS DADOS DE UM ESTUDANTE NO FORMATO TEXTO, SERÁ UTILIZADA A SAÍDA DA FERRAMENTA DE DADOS DE ESTUDANTE 
    # COMO ENTRADA DESTA FERRAMENTA DE PERFIL ACADÊMICO.
    #
    # ASSIM, A ENTRADA DESTE MÉTODO _run SERÁ O TEXTO COM OS DADOS DO ESTUDANTE, COMO CONTEXTO DE UM PROMPT, QUE SERÁ USADO 
    # PARA INFORMAR A MANEIRA COMO O PERFIL ACADÊMICO DEVE SER GERADO.
    # 
    def _run(self, input: str) -> str:        
        
        parseador = JsonOutputParser(pydantic_object=ExtratorPerfilAcademicoDeEstudante)    
        
        template = PromptTemplate(
                                    template = """ 
                                                    CONTEXTO:                            
                                                    Você é uma consultora de carreiras
                                                                                
                                                    Esta ferramenta utiliza os dados do estudante para gerar um perfil acadêmico detalhado.
                                                    
                                                    OBJETIVO:
                                                        - Criar o perfil acadêmico de um estudante utilizando os dados do estudante fornecidos 
                                                        
                                                        ENTRADA:
                                                        -------------------------
                                                        {dados_do_estudante}
                                                        -------------------------
                                                    
                                                    ESTILO:
                                                    Precisa indicar com detalhes, riqueza, mas direta ao ponto.
                                                                                
                                                    PASSOS:
                                                        - Formate o estudante para o seu perfil acadêmico.
                                                        - Com os dados, identifique as opções de universidades sugeridas e cursos compatíveis com o interesse do aluno.
                                                        - Destaque o perfil do aluno, dando ênfase, principalmente, naquilo que faz interesse nas instituições de interesse
                                                        do aluno.                                                      
                                            
                                                    FORMATO DE SAIDA:
                                                    {formato_saida} 
                                                    
                                                    - Saída em português.
                                                """,
                                    input_variables = ["dados_do_estudante"],
                                    partial_variables = {"formato_saida": parseador.get_format_instructions()}
                                 )
        
        cadeia = template | self.llm | parseador        
        
        print("\nEntrada Perfil Acadêmico\n", input)
        
        resposta = cadeia.invoke({"dados_do_estudante": input})
        
        print('Resposta Perfil Acadêmico\n',resposta)
        
        return resposta


<b>2) Instanciando as Ferramentas que a LLM precisa usar</b>

In [5]:
from langchain.agents import Tool

class Tools:
        
        def __init__(self,llm:ChatOpenAI):
                
                dados_de_estudante = DadosDeEstudante()
                perfil_academico = PerfilAcademico(llm) # INSTANCIANDO O OBJETO DA MINHA FERRAMENTA

                # MATRIZ DE FERRAMENTAS (CONJUNTO DE FERRAMENTAS)
                self.tools = [
                        # Instanciando ferramentas
                        Tool(
                                name=dados_de_estudante.name,
                                func=dados_de_estudante.run,
                                description=dados_de_estudante.description                                          
                        ),
                        
                        # SEGUNDA FERRAMENTA. 
                        # SE VIRA PARA PEGAR OS DADOS DO ESTUDANTE E GERAR O PERFIL ACADÊMICO
                        Tool(
                                name=perfil_academico.name,
                                func=perfil_academico.run,
                                description=perfil_academico.description                                                      
                        )
                ]

### <b>CRIAÇÃO DO AGENTE DE FERRAMENTAS</b>

<b>3) Informando para a LLM as ferramentas que eu tenho</b> 
<ul><li>Para isso, é necessário criar um agente com as ferramentas</li></ul><br/>
<b>Que outro prompt eu posso utilizar, se eu não tiver o openai functions na criação do agente, ou seja, esteja utlizando outra LLM ?</b><br/><br/>
<b>Abordagem e metodologia</b><br/>
<ul><li>O framework ReAct promove uma fusão entre raciocínio detalhado e ações práticas dentro de um fluxo de trabalho interativo, permitindo que modelos de linguagem atuem de forma adaptativa e contextual. Esse método é particularmente valioso em domínios que exigem verificação de fatos e respostas informadas por dados atualizados.</li></ul>

In [6]:
#from langchain.agents import create_openai_tools_agent
from langchain.agents import create_react_agent
from langchain import hub
from dotenv import load_dotenv
from os import getenv
import warnings

warnings.filterwarnings("ignore")

class AgenteReAct:
    
    def __init__(self):
      
      load_dotenv()

      llm = ChatOpenAI(
                        model="gpt-4.1-mini", # TIVE QUE TROCAR PARA UM MODELO MENOR, POR CAUSA DO ERRO ABAIXO
                                              #
                                              #  BadRequestError: Error code: 400 - {'error': {'message': "Unsupported parameter: 'stop' is not supported with this model.", 
                                              #  'type': 'invalid_request_error', 'param': 'stop', 'code': 'unsupported_parameter'}}
                                              #
                        api_key=getenv("API_KEY") 
                      )
      
      # INSTANCIANDO AS FERRAMENTAS
      self.tools = Tools(llm).tools
        
      # PROMPT DE INICIALIZAÇÃO PARA INFORMAR PARA A LLM SOBRE A FERRAMENTA.
      # openai-functions-agent É UM PROMPT PRONTO PARA AGENTES QUE UTILIZAM FUNÇÕES (TOOLS)
      #prompt=(hub.pull(owner_repo_commit="hwchase17/openai-functions-agent"))
      
      # react-agent É UM PROMPT PRONTO PARA AGENTES QUE UTILIZAM O PARADIGMA ReAct (Raciocínio e Ação)
      prompt=(hub.pull(owner_repo_commit="hwchase17/react"))
      

      # CRIANDO UM AGENTE COM AS FERRAMENTAS
      self.agente = create_react_agent(
                                        llm=llm, # INFORMA A LLM QUE VAI SER USADA PELO AGENTE
                                        tools=self.tools, # PASSANDO PARA A LLM A FERRAMENTA QUE ELA PODE USAR. INSTÂNCIA DA FERRAMENTA
                                        prompt=prompt  
                                                          # JÁ EXISTEM PROMPTS PRONTOS NO REPOSITÓRIO DO LANGSMITH, DE ACORDO COM O TIPO DE FERRAMENTA. 
                                                          # Para agente react (https://smith.langchain.com/hub/hwchase17/react)  
                                      )
      
      """ self.agente = create_openai_tools_agent(
                                                llm=llm, # INFORMA A LLM QUE VAI SER USADA PELO AGENTE
                                                tools=self.tools, # PASSANDO PARA A LLM A FERRAMENTA QUE ELA PODE USAR. INSTÂNCIA DA FERRAMENTA
                                                prompt=prompt  
                                                                  # JÁ EXISTEM PROMPTS PRONTOS NO REPOSITÓRIO DO LANGSMITH, DE ACORDO COM O TIPO DE FERRAMENTA. 
                                                                          # Para agente de função (https://smith.langchain.com/hub/hwchase17/openai-functions-agent)
                                                                                
                                             ) """

      print(prompt)

<b>4) Executando o agente com as ferramentas</b>

"Marcos" não foi chamado ainda, corre-se o risco de alucinação. Ver o comentário na criação da ferramenta DadosDeEstudante

In [ ]:
from langchain.agents import AgentExecutor

agente = AgenteReAct()

executor = AgentExecutor(
                            agent=agente.agente, # O AGENTE QUE VAI SER USADO
                            tools=agente.tools, # FERRAMENTAS QUE O AGENTE PODE OU NÃO USAR
                            verbose=True
                        )

for pergunta in [
                    "Compare o perfil acadêmico da Ana com o da Bianca.", # NECESSÁRIO RECEBER OS DADOS DA PRIMEIRA FERRAMENTA, DADOS DE ESTUDANTE, COMO ENTRADA.
                    "Tenho sentido Ana desanimada com cursos de Matemática. Seria uma boa, parear ela com o Marcos?" # "Marcos" não foi chamado ainda,
                                                                                                                     # corre-se o risco de alucinação. 
                                                                                                                     # Ver o comentário na criação da 
                                                                                                                     # ferramenta DadosDeEstudante 
                                                                                                                                                                                                                                      
                ]:
    
    print('\nPergunta: ', pergunta,"\n")
    
    print()
    resposta = executor.invoke({"input": pergunta})
    print(resposta)
    
    

Inicializando ferramenta Dados de Estudante
Inicializando ferramenta Perfil Acadêmico
input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'

Pergunta:  Compare o perfil acadêmico da Ana co